In [ ]:

# ==========================================
# TARGET DATA SETUP
# ==========================================
t_valid = np.array([-40.0, -23.5, -7.0, 9.5, 26.0, 42.5, 59.0, 75.5, 92.0, 108.5, 125.0])
t_reg = np.array([-7.0, 9.5, 26.0, 42.5, 59.0, 75.5, 92.0, 108.5])
v_reg = np.array([0.900, 0.864, 0.810, 0.780, 0.751, 0.722, 0.679, 0.610])

# 1. CALCULATE LINEAR REGRESSION TARGET
slope_target, intercept_target = np.polyfit(t_reg, v_reg, 1)
v_fit_plot_target = slope_target * t_valid + intercept_target

print("========================================")
print("--- TARGET IDEAL CURVE ---")
print(f"Absolute Slope:  {slope_target*1000:.3f} mV/°C")
print(f"Voltage @ 25°C:  {(slope_target * 25.0 + intercept_target)*1000:.1f} mV")
print("========================================\n")

# ==========================================
# LOAD DATA 1: RAW CORNERS
# ==========================================
df_corners = pd.DataFrame()
corner_cols = {}
try:
    df_corners = pd.read_csv('VTEMP_CORNERS_NEWVARS', skiprows=1, sep=r'\s+')
    corner_cols['TT'] = next((c for c in df_corners.columns if 'TT.1:' in c), None)
    corner_cols['SS'] = next((c for c in df_corners.columns if 'SS.1:' in c), None)
    corner_cols['FF'] = next((c for c in df_corners.columns if 'FF.1:' in c), None)
except FileNotFoundError:
    print("Error: 'VTEMP_CORNERS_NEWVARS' not found. Ensure the file is in the same directory.")

# ==========================================
# LOAD DATA 2: FINAL CALIBRATED CORNERS
# ==========================================
df_calibrated = pd.DataFrame()
tt_cols = ss_cols = ff_cols = []
new_data_file = 'Vtemp_corners_calibrated_final'

try:
    with open(new_data_file, 'r') as f:
        lines = f.readlines()
        
    headers = []
    all_floats = []
    
    for line in lines:
        line = line.strip()
        if line.startswith('XVAL'):
            headers = line.split()
        else:
            for item in line.split():
                try:
                    all_floats.append(float(item))
                except ValueError:
                    pass
    
    num_cols = len(headers)
    if num_cols > 0 and len(all_floats) > 0:
        valid_start_idx = 0
        for i in range(len(all_floats)):
            if all_floats[i] == -40.0:
                valid_start_idx = i
                break
                
        clean_floats = all_floats[valid_start_idx:]
        num_rows = len(clean_floats) // num_cols
        data_2d = [clean_floats[i*num_cols : (i+1)*num_cols] for i in range(num_rows)]
        
        df_calibrated = pd.DataFrame(data_2d, columns=headers)
        
        tt_cols = [c for c in df_calibrated.columns if 'TT' in c]
        ss_cols = [c for c in df_calibrated.columns if 'SS' in c]
        ff_cols = [c for c in df_calibrated.columns if 'FF' in c]

except FileNotFoundError:
    pass
except Exception as e:
    pass

# ==========================================
# PLOTTING
# ==========================================
fig, axes = plt.subplots(3, 1, figsize=(10, 18))
colors = {'TT': 'black', 'SS': 'blue', 'FF': 'green'}
markers = {'TT': 'o', 'SS': 's', 'FF': '^'}

# ------------------------------------------
# PLOT 1: RAW OUTPUT
# ------------------------------------------
axes[0].plot(t_valid, v_fit_plot_target * 1000, color='red', linewidth=3, linestyle='--', label='Ideal Target (Calculated Linear Fit)')

if not df_corners.empty:
    for corner, col in corner_cols.items():
        if col:
            axes[0].plot(df_corners['XVAL'], df_corners[col] * 1000, color=colors[corner], linewidth=2, marker=markers[corner], markersize=4, label=f'Raw Vtemp ({corner})')

axes[0].set_title('1. Raw Simulator Output (Zero Offsets Applied)', fontweight='bold')
axes[0].set_xlabel('Temperature (°C)')
axes[0].set_ylabel('Voltage (mV)')
axes[0].grid(True, linestyle='--', alpha=0.6)
axes[0].legend(loc='best')

# ------------------------------------------
# PLOT 2: MATHEMATICALLY TRANSFORMED (WITH CALCULATED GAIN & OFFSET)
# ------------------------------------------
axes[1].plot(t_valid, v_fit_plot_target * 1000, color='red', linewidth=3, linestyle='--', label='Ideal Target (Calculated Linear Fit)')

if not df_corners.empty:
    t_sim_raw = df_corners['XVAL'].values
    mask = (t_sim_raw >= -7.0) & (t_sim_raw <= 108.5)
    
    for corner, col in corner_cols.items():
        if col:
            v_sim_raw = df_corners[col].values
            
            # Calculate the linear regression for the current corner's raw simulator data
            slope_sim, intercept_sim = np.polyfit(t_sim_raw[mask], v_sim_raw[mask], 1)
            
            # Calculate required mathematically perfect Gain and Offset
            req_gain = slope_target / slope_sim
            req_offset = intercept_target - (req_gain * intercept_sim)
            
            # Apply the transformation: V_transformed = (V_raw * Gain) + Offset
            transformed_v = (v_sim_raw * req_gain) + req_offset
            
            # Plot the transformed curve and dynamically update the legend with the calculated values
            axes[1].plot(t_sim_raw, transformed_v * 1000, color=colors[corner], linewidth=2, marker=markers[corner], markersize=4, 
                         label=f'Transformed {corner} (Gain: {req_gain:.2f}x | Offset: {req_offset*1000:.1f} mV)')

axes[1].set_title('2. Mathematically Transformed Output (Calculated Ideal Gains & Offsets)', fontweight='bold')
axes[1].set_xlabel('Temperature (°C)')
axes[1].set_ylabel('Voltage (mV)')
axes[1].grid(True, linestyle='--', alpha=0.6)
axes[1].legend(loc='best')

# ------------------------------------------
# PLOT 3: FINAL CALIBRATED DATA (FROM CSV)
# ------------------------------------------
axes[2].plot(t_valid, v_fit_plot_target * 1000, color='red', linewidth=6, linestyle='-', alpha=0.3, label='Ideal Target')

if not df_calibrated.empty:
    if tt_cols:
        axes[2].plot(df_calibrated['XVAL'], df_calibrated[tt_cols[0]] * 1000, color=colors['TT'], linewidth=2, marker=markers['TT'], markersize=5, label='Calibrated TT Simulator')
    if ss_cols:
        axes[2].plot(df_calibrated['XVAL'], df_calibrated[ss_cols[0]] * 1000, color=colors['SS'], linewidth=2, marker=markers['SS'], markersize=5, label='Calibrated SS Simulator')
    if ff_cols:
        axes[2].plot(df_calibrated['XVAL'], df_calibrated[ff_cols[0]] * 1000, color=colors['FF'], linewidth=2, marker=markers['FF'], markersize=5, label='Calibrated FF Simulator')
        
        # --- NEW CODE: Print table for FF corner ---
        print("========================================")
        print("--- FINAL CALIBRATED POINTS (FF CORNER) ---")
        print(f"{'Temperature (°C)':<18} | {'Voltage (mV)':<18}")
        print("-" * 40)
        for t, v in zip(df_calibrated['XVAL'], df_calibrated[ff_cols[0]]):
            print(f"{t:<18.1f} | {v * 1000:<18.3f}")
        print("========================================\n")

axes[2].set_title('3. Final Calibrated Output Data vs Ideal Target', fontweight='bold')
axes[2].set_xlabel('Temperature (°C)')
axes[2].set_ylabel('Voltage (mV)')
axes[2].grid(True, linestyle='--', alpha=0.6)
axes[2].legend(loc='best')

plt.tight_layout()
plt.show()